# 🕵️ Dia 69 — Golpes românticos e de investimento por relacionamento

**Investigador Cyber 90 Dias** · Fase 5 · Rank Jutsu `A`

> Baseado no *Day 69* de [cyber-investigator-skills](https://github.com/thiefdroppers/cyber-investigator-skills/blob/main/curriculum/phase-5-fraud-social-engineering/day-69.md) (*Romance and relationship-investment scams*), © ThiefDroppers (MIT).
> Notebook, camada BR e Camada 8 por Deivison Lourenço ([@Ridd1kulusC0d3r](https://github.com/Ridd1kulusC0d3r)).

---

### 🎬 Onde paramos

Sexta, 23 de outubro. Relato R-CRED-09, da credminas.example: R$ 21,5 mil num "robô de investimento em ouro" apresentado por "Ricardo" depois de quase dois meses de conversa. O último Pix foi para **PIXFICT-L04**, a mesma chave do e-mail da Gráfica Jatobá.

Rafa: *"Primeiro, a história dela, na ordem. E ninguém aqui vai chamar essa senhora de ingênua."*

Neste notebook você vai:

1. Ler a linha do tempo no **formato do TimelineJS** e conferir os estágios obrigatórios.
2. Calcular o **primeiro ponto de exposição** e cada ponto em que o **dinheiro se moveu**.
3. Jogar o **quiz do estágio** com os casos "Marcus" e "Ricardo".
4. Passar a análise pelo **verificador de linguagem revitimizante** e gerar o diagrama `timeline`.

🐍 Só biblioteca padrão do Python. Nada para instalar. Rode as células de cima para baixo (Shift+Enter). Todos os dados são **fictícios** (domínios `.example`, telefones `+55 00 00000-00NN`, chaves Pix `PIXFICT-*`).

## 1. A linha do tempo do R-CRED-09

No mesmo formato do modelo de planilha do TimelineJS (colunas Year, Month, Day, Display Date, Headline, Text, Group...). Embutida para rodar offline.

In [ ]:
TIMELINE_CSV = 'Year,Month,Day,Time,End Year,End Month,End Day,End Time,Display Date,Headline,Text,Media,Media Credit,Media Caption,Media Thumbnail,Type,Group,Background\n2026,6,2,,,,,,Dia 1,Pedido de amizade e elogios,"Perfil \'Ricardo\', diz ser engenheiro em plataforma de petróleo. Contato numa rede social. [sinal: contato não solicitado]",,,,,,1 Contato,\n2026,6,4,,,,,,Dia 3,Vai para o WhatsApp,Ricardo pede para conversar \'num lugar mais reservado\' (+55 00 00000-0031). [sinal 2],,,,,,2 Sai da plataforma,\n2026,6,5,,,,,,Dias 4 a 25,Mensagens diárias e planos juntos,"Bom-dia todo dia, histórias da plataforma, fala em \'morar juntos quando eu desembarcar\'. Declara amor no dia 9. [sinais 1 e 4]",,,,,,3 Construção de confiança,\n2026,6,15,,,,,,Dia 14,Câmera \'quebrada\',Pedido de chamada de vídeo recusado: \'na plataforma não pode câmera\'. [sinal 3],,,,,,3 Construção de confiança,\n2026,6,27,,,,,,Dia 26,A plataforma do tio,Ricardo fala de um \'robô de investimento em ouro\' em ouro-facil.example e manda um print com lucros. [sinal 7],,,,,,5 O pedido,\n2026,6,28,,,,,,Dia 27,Primeiro depósito,Depósito de R$ 500 por Pix; o painel mostra R$ 640 dois dias depois. [sinal 8],,,,,,5 O pedido,\n2026,7,10,,,,,,Dias 39 a 55,Nível VIP,"Mais R$ 21.000 em quatro Pix, o último para PIXFICT-L04 (fictícia). Segunda chamada de vídeo recusada. [sinais 3 e 6]",,,,,,6 Escalada,\n2026,8,2,,,,,,Dia 62,Saque bloqueado,"Para sacar, exigem \'imposto de liberação\' de 18%. Ricardo pede segredo: \'sua família não entenderia\'. [sinais 8 e 9]",,,,,,7 Saque bloqueado,\n2026,8,30,,,,,,Dia 90,Golpe da recuperação,Um \'escritório de recuperação de ativos\' oferece recuperar tudo mediante taxa adiantada. [oferta de recuperação não solicitada],,,,,,8 Golpe da recuperação,\n'

import csv, io, re

eventos = list(csv.DictReader(io.StringIO(TIMELINE_CSV)))
for e in eventos:
    # "Group" é a coluna do TimelineJS que agrupa as faixas: usamos para o estágio.
    print(f"{e['Display Date']:12} | {e['Group']:24} | {e['Headline']}")

## 2. Os estágios e os checkpoints do original

O checkpoint pede **pelo menos uma entrada** nos estágios 1 (Contato), 5 (O pedido) e 7 (Saque bloqueado), e que **cada sinal marcado aponte para um trecho da fonte**. A célula confere as duas coisas.

In [ ]:
ESTAGIOS = {1: "Contato", 2: "Sai da plataforma", 3: "Construção de confiança", 4: "Motivo para não se encontrarem",
            5: "O pedido", 6: "Escalada", 7: "Saque bloqueado", 8: "Golpe da recuperação"}

def numero_do_estagio(grupo):
    """'5 O pedido' → 5. A planilha guarda o número na frente do nome."""
    m = re.match(r"\s*(\d)", grupo)
    return int(m.group(1)) if m else None

presentes = {numero_do_estagio(e["Group"]) for e in eventos}
for obrigatorio in (1, 5, 7):
    ok = obrigatorio in presentes
    print(f"Estágio {obrigatorio} ({ESTAGIOS[obrigatorio]}): {'✅' if ok else '❌ falta entrada'}")

# Cada evento precisa trazer a marcação de sinal entre colchetes no fim do Text: [sinal 3], [sinais 3 e 6]...
sem_marcacao = [e["Headline"] for e in eventos if not re.search(r"\[[^\]]+\]\s*$", e["Text"])]
print("Eventos sem marcação de sinal:", sem_marcacao or "nenhum ✅")

## 3. Primeira exposição e o caminho do dinheiro

A primeira exposição é o primeiro evento com uma checagem que a vítima podia fazer sozinha: **chamada de vídeo recusada** (sinal 3), **plataforma desconhecida** (sinal 7, no Brasil: lista de alertas da CVM) ou foto encontrada sob outro nome (sinal 10).

Os valores em reais são extraídos do texto com regex. Atenção: o painel da plataforma mostra valores **fictícios** (o "lucro"); só entram na soma os valores que **saíram** da vítima.

In [ ]:
# Sinais que correspondem a uma checagem que a vítima podia fazer sozinha
SINAIS_DE_CHECAGEM = {"3": "chamada de vídeo ao vivo e sem combinar",
                      "7": "pesquisar a plataforma (no Brasil: alertas da CVM)",
                      "10": "busca reversa de imagem"}

def sinais(texto):
    """Extrai os números de sinal da marcação final, ex.: '[sinais 3 e 6]' → ['3', '6']."""
    marcacao = re.search(r"\[([^\]]+)\]\s*$", texto)
    return re.findall(r"\d+", marcacao.group(1)) if marcacao else []

primeira = next(((e, s) for e in eventos for s in sinais(e["Text"]) if s in SINAIS_DE_CHECAGEM), None)
if primeira:
    e, s = primeira
    print(f"🔎 Primeira exposição: {e['Display Date']} — {e['Headline']}")
    print(f"   checagem: {SINAIS_DE_CHECAGEM[s]}")

def valor_em_reais(trecho):
    """'R$ 21.000' → 21000.0 (formato brasileiro: ponto de milhar, vírgula decimal)."""
    return float(trecho.replace(".", "").replace(",", "."))

print("\n💸 Onde o dinheiro se moveu (estágios 5 a 8):")
total = 0.0
for e in eventos:
    n = numero_do_estagio(e["Group"])
    if n and n >= 5:
        # Só valores precedidos de verbo de saída: "Depósito de", "Mais", não o que "o painel mostra"
        saidas = re.findall(r"(?:Depósito de|Mais) R\$ ([\d.,]+)", e["Text"])
        soma = sum(valor_em_reais(v) for v in saidas)
        total += soma
        marca = f"saiu R$ {soma:,.0f}".replace(",", ".") if soma else "pedido sem valor pago no relato"
        print(f"   {e['Display Date']:12} {ESTAGIOS[n]:22} {marca}")
print(f"   TOTAL que saiu da vítima: R$ {total:,.0f}".replace(",", "."))

## 4. 🎮 Quiz do estágio

Para cada evento, diga o número do estágio (1 a 8). Mistura o caso "Marcus" (original) e o "Ricardo" (R-CRED-09). `MODO_JOGO = False` mostra só o gabarito.

In [ ]:
MODO_JOGO = True

QUIZ = [
    ("🇺🇸", "SMS de 'número errado'; o remetente pede desculpas e continua conversando", 1),
    ("🇺🇸", "Marcus menciona a 'arbitragem de ouro' do tio e a plataforma aurum-desk.example", 5),
    ("🇺🇸", "Saque bloqueado até pagar 'liberação de imposto' de 15%", 7),
    ("🇺🇸", "'Empresa de recuperação em blockchain' oferece rastrear os fundos por uma taxa", 8),
    ("🇧🇷", "Ricardo pede para conversar 'num lugar mais reservado' no WhatsApp", 2),
    ("🇧🇷", "Mensagens diárias e 'morar juntos quando eu desembarcar'", 3),
    ("🇧🇷", "Mais R$ 21.000 em quatro Pix para chegar ao nível VIP", 6),
    ("🇧🇷", "'Na plataforma de petróleo não pode câmera'", 4),
]
acertos = 0
for bandeira, evento, certo in QUIZ:
    print(f"\n{bandeira} {evento}")
    if MODO_JOGO:
        r = ""
        while not (r.isdigit() and 1 <= int(r) <= 8):
            r = input("   Estágio (1-8): ").strip()
        acertos += int(r) == certo
        print("   ✅" if int(r) == certo else "   ❌")
    print(f"   Gabarito: {certo} — {ESTAGIOS[certo]}")
if MODO_JOGO:
    print(f"\n🏁 {acertos}/{len(QUIZ)}")

> 💡 O evento "na plataforma não pode câmera" pode ser lido como estágio 3 (é uma recusa de vídeo durante a construção de confiança, e a planilha do caso o marca assim) **ou** 4 (é o motivo para não se encontrarem). O diagrama do original liga a checagem do vídeo aos dois estágios. O que não pode faltar é o **sinal 3** marcado.

## 5. Verificador de linguagem revitimizante

Escreva a sua análise de dois parágrafos no formulário. O verificador procura os termos que o checkpoint do original proíbe e outros comuns em português. Ele não substitui a revisão humana; pega o óbvio.

In [ ]:
#@title Sua análise (2 parágrafos) { display-mode: "form" }
paragrafo_1 = "A primeira checagem que teria exposto o golpe é a do dia 14: a recusa da chamada de vídeo, somada ao contato não solicitado do dia 1. No dia 26, uma consulta aos alertas da CVM sobre ouro-facil.example teria sido a segunda chance."  #@param {type:"string"}
paragrafo_2 = "O dinheiro saiu em dois momentos induzidos pelo golpista: o depósito inicial de R$ 500 (dia 27) e os quatro Pix que somaram R$ 21.000 para o nível VIP (dias 39 a 55, o último para PIXFICT-L04). Depois ele tentou mais duas cobranças: o imposto de liberação (dia 62) e a falsa recuperação mediante taxa (dia 90)."  #@param {type:"string"}

PROIBIDOS = {
    r"\bingênu[oa]s?\b": "descreva a técnica do golpista, não o caráter da vítima",
    r"\bgananci\w*": "o 'ganho' foi fabricado pelo painel falso",
    r"\b(pig butchering|abate do porco|porco)\b": "use 'golpe de investimento por relacionamento'",
    r"\bcaiu (no|em|n[ao]) golpe\b": "prefira 'foi alvo de' ou 'foi enganada por'",
    r"\bdeixou[- ]se enganar\b": "o sujeito da frase é o golpista",
    r"\bboba?\b|\btrouxa\b|\botári[oa]\b": "linguagem depreciativa",
    r"\bnaive\b|\bgreedy\b": "termos que o checkpoint do original proíbe",
}

def revisar(texto):
    """Devolve a lista de (trecho, dica) encontrados."""
    achados = []
    for padrao, dica in PROIBIDOS.items():
        for m in re.finditer(padrao, texto, flags=re.I):
            achados.append((m.group(), dica))
    return achados

analise = paragrafo_1 + "\n\n" + paragrafo_2
problemas = revisar(analise)
for trecho, dica in problemas:
    print(f"⚠️  '{trecho}' → {dica}")
print("✅ Nenhum termo revitimizante encontrado." if not problemas else f"\n{len(problemas)} ajuste(s) antes de entregar.")

# Teste do próprio verificador com uma frase ruim (para você ver que ele funciona)
print("\nTeste:", revisar("A vítima, ingênua, caiu no golpe por ganância."))

## 6. 📝 O diagrama `timeline` e o arquivo do dia

In [ ]:
from pathlib import Path

linhas = ["```mermaid", "timeline", "    title Relato R-CRED-09 (ficticio), por estagio"]
estagio_atual = None
for e in eventos:
    n = numero_do_estagio(e["Group"])
    if n != estagio_atual:                       # nova seção a cada troca de estágio
        linhas.append(f"    section {ESTAGIOS[n]}")
        estagio_atual = n
    # O mermaid timeline usa ':' como separador: tiramos ':' do título por segurança
    titulo = e["Headline"].replace(":", " -")
    marcas = ", ".join(sinais(e["Text"]))
    # eventos sem número de sinal (abertura, recuperação) levam a marcação em texto
    marcas = f"sinais {marcas}" if marcas else re.search(r"\[([^\]]+)\]", e["Text"]).group(1)
    linhas.append(f"        {e['Display Date']} : {titulo} ({marcas})")
linhas.append("```")
diagrama = "\n".join(linhas)

texto = ("# Análise — R-CRED-09 (Dia 69, Operação Isca, fictício)\n\n" + diagrama + "\n\n"
         + analise + "\n\n"
         "Nota de grafo (Dia 68): R-CRED-09 e BEC-B passam a compartilhar PIXFICT-L04 (forte), "
         "mas nenhum dos dois se liga ao agrupamento V/E. Hipótese: conta de passagem usada por mais de um esquema.\n")
Path("notas").mkdir(exist_ok=True)
saida = Path("notas/dia69-analise.md")
saida.write_text(texto, encoding="utf-8")
print(texto)
try:
    from google.colab import files
    files.download(str(saida))
except ImportError:
    print("(Fora do Colab: o arquivo ficou em notas/)")

### 🧠 Camada 8

Custo afundado (Arkes e Blumer) move a escalada; a vergonha move o silêncio e abre a porta para o golpe da recuperação. Chamar a vítima de ingênua é **erro fundamental de atribuição** (Ross): explica pelo caráter o que a situação construiu. Os verbos do relatório são do golpista.

## ✅ Antes de fechar o Dia 69

- [ ] Estágios 1, 5 e 7 presentes; todo evento com sinal marcado
- [ ] Primeira exposição identificada (dia 14) e o caminho do dinheiro somado
- [ ] Verificador de linguagem sem alertas na sua análise
- [ ] R-CRED-09 acrescentado ao grafo do Dia 68 (L-04 liga BEC-B e R-CRED-09, mas não ao agrupamento V/E)

**Amanhã (Dia 70):** o dia em que o investigador **para**. Indicadores de aliciamento e tráfico, o que eles não dizem e para onde se encaminha.

---
<sub>Baseado em cyber-investigator-skills por ThiefDroppers (MIT) · Investigador Cyber 90 Dias por Deivison Lourenço</sub>